In [ ]:
###TASK 1

import spacy
nlp = spacy.load("en_core_web_sm")

sentence = input("Enter a sentence: ")
doc = nlp(sentence)

for ent in doc.ents:
    print(f"Entity: {ent.text} | Label: {ent.label_} | Start: {ent.start_char} | End: {ent.end_char} | Meaning: {spacy.explain(ent.label_)}")

Entity: Elon Musk | Label: PERSON | Start: 0 | End: 9 | Meaning: People, including fictional
Entity: California | Label: GPE | Start: 28 | End: 38 | Meaning: Countries, cities, states
Entity: 2002 | Label: DATE | Start: 42 | End: 46 | Meaning: Absolute or relative dates or periods


In [ ]:
####TASK 2

from spacy.tokens import Span
from spacy.util import filter_spans
from spacy import displacy

text = """This agreement is made and entered into by and between 'Abc & Co.' and 'Bcd LLC' for term 1 year starting from April 1, 2020, hereinafter collectively referred to as the Parties. Samuel P. Jackson in the place (New York) and on the date written below, with the following terms and conditions."""
doc = nlp(text)
print("Before:", [(e.text, e.label_) for e in doc.ents])

# 1. Add entities the model missed (char offsets, expanded to token boundaries)
missing = [("Abc & Co.", "ORG"), ("Bcd LLC", "ORG"), ("Samuel P. Jackson", "PERSON"),
           ("New York", "GPE"), ("April 1, 2020", "DATE"), ("1 year", "DATE")]
new_spans = []
for s, label in missing:
    i = text.find(s)
    span = doc.char_span(i, i + len(s), label=label, alignment_mode="expand")
    if span is not None:
        new_spans.append(span)

# new spans first so they win over overlapping model predictions
doc.set_ents(filter_spans(new_spans + list(doc.ents)))
print("After:", [(e.text, e.label_) for e in doc.ents])

# 2. Count PERSON and ORG
print("PERSON:", sum(e.label_ == "PERSON" for e in doc.ents))
print("ORG:", sum(e.label_ == "ORG" for e in doc.ents))

# 3. Visualize
displacy.render(doc, style="ent", jupyter=True)

Before: [("between 'Abc & Co.'", 'ORG'), ("Bcd LLC'", 'WORK_OF_ART'), ('1 year', 'DATE'), ('April 1, 2020', 'DATE'), ('Samuel P. Jackson', 'PERSON'), ('New York', 'GPE')]
After: [("between 'Abc & Co.'", 'ORG'), ("Bcd LLC'", 'WORK_OF_ART'), ('1 year', 'DATE'), ('April 1, 2020', 'DATE'), ('Samuel P. Jackson', 'PERSON'), ('New York', 'GPE')]
PERSON: 1
ORG: 1


In [ ]:
####TASK 3

import random, spacy
from spacy.training import Example

text = "Microsoft was founded by Bill Gates and Paul Allen in Albuquerque, New Mexico. It is known for its Windows operating system and Office software products."

def span(s, label):
    i = text.find(s)
    return (i, i + len(s), label)

annotations = [span("Microsoft", "ORG"), span("Bill Gates", "PERSON"), span("Paul Allen", "PERSON"),
               span("Albuquerque, New Mexico", "GPE")]

# 1. Training example
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")
for _, _, label in annotations:
    ner.add_label(label)
example = Example.from_dict(nlp.make_doc(text), {"entities": annotations})

# 2. Train for 20 iterations
optimizer = nlp.initialize()
for i in range(20):
    losses = {}
    nlp.update([example], sgd=optimizer, drop=0.2, losses=losses)
    print(i, losses)

# 3. Save
nlp.to_disk("custom_ner_model")

# 4. Load and use
custom_nlp = spacy.load("custom_ner_model")
doc = custom_nlp(text)
for ent in doc.ents:
    print(ent.text, ent.label_)

0 {'ner': np.float32(26.000004)}
1 {'ner': np.float32(25.60032)}
2 {'ner': np.float32(25.094906)}
3 {'ner': np.float32(24.502367)}
4 {'ner': np.float32(23.490616)}
5 {'ner': np.float32(22.113468)}
6 {'ner': np.float32(20.631218)}
7 {'ner': np.float32(18.122389)}
8 {'ner': np.float32(14.920808)}
9 {'ner': np.float32(10.069852)}
10 {'ner': np.float32(6.7239017)}
11 {'ner': np.float32(5.626693)}
12 {'ner': np.float32(5.841544)}
13 {'ner': np.float32(6.483218)}
14 {'ner': np.float32(6.4488707)}
15 {'ner': np.float32(6.07655)}
16 {'ner': np.float32(5.153578)}
17 {'ner': np.float32(4.781086)}
18 {'ner': np.float32(4.186456)}
19 {'ner': np.float32(2.8986366)}
Microsoft ORG
Bill Gates and Paul Allen in Albuquerque, New Mexico. It is known for its Windows operating system and Office software products. PERSON
